# Imports

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Current Registry

In [30]:
registry_df = pd.read_csv("current_rental_registrations_251001.csv")

In [31]:
registry_df["formatted_address"] = registry_df["RegisteredAddress"].str.replace(r"\s+,", ",", regex=True)

In [32]:
registry_df["formatted_address"] = registry_df["formatted_address"].astype(str).str.upper().str.strip()

In [33]:
# Remove unit numbers before the first comma (e.g., " AVE 5," → " AVE,")
registry_df["formatted_address"] = registry_df["formatted_address"].str.replace(
    r"\s+\d+(?=,)", "", regex=True
).str.strip()


In [34]:
import re

def clean_units(address):
    # Match pattern: [street] [unit], [city], MA [ZIP]
    match = re.match(r"^(.*\b(?:ST|AV|AVE|RD|BLVD|PL|CT|DR|TER|WAY|LN|SQ|TE|CIR|PKWY|PLZ|HWY))\s+[A-Z0-9\-]+, (.+?, MA \d{5})$", address)
    if match:
        return f"{match.group(1)}, {match.group(2)}"
    return address

registry_df["formatted_address"] = registry_df["formatted_address"].apply(clean_units)


In [35]:
registry_df['formatted_address'].sample(30)

19572        14-18 ADAMS ST, CHARLESTOWN, MA 02129
16817    375 BUNKER HILL ST, CHARLESTOWN, MA 02129
17124        84 GREENWOOD ST, DORCHESTER, MA 02121
2147               277 BEACON ST, BOSTON, MA 02116
20701          111 PIERCE AV, DORCHESTER, MA 02122
34213       80-82 HUNNEWELL AV, BRIGHTON, MA 02135
35361           11 JOSEPH ST, DORCHESTER, MA 02124
12526           1 WISE ST, JAMAICA PLAIN, MA 02130
6110         17 AINSWORTH ST, ROSLINDALE, MA 02131
19156             80 RUTLAND ST, ROXBURY, MA 02118
5095             19-21 ROYAL ST, ALLSTON, MA 02134
1314        16 OAKVIEW TE, JAMAICA PLAIN, MA 02130
38371      664 MASSACHUSETTS AV, ROXBURY, MA 02118
12397       72 WILLOWWOOD ST, DORCHESTER, MA 02124
16790    306 BUNKER HILL ST, CHARLESTOWN, MA 02129
4233           30 RIDGEVIEW AV, MATTAPAN, MA 02126
33648       94 HILLSIDE ST, MISSION HILL, MA 02120
15738            17 BRENT ST, DORCHESTER, MA 02124
4321            1323 RIVER ST, HYDE PARK, MA 02136
31582           54 GREEN ST, CH

# Assessor Dataset

In [36]:
assessment_df = pd.read_csv("fy2025-property-assessment-data_12_30_2024.csv", dtype={21: str}, low_memory=False)

In [37]:
def combine_street_numbers(row):
    try:
        st_num = str(int(float(row['ST_NUM']))) if pd.notnull(row['ST_NUM']) else ""
        st_num2 = str(int(float(row['ST_NUM2']))) if pd.notnull(row['ST_NUM2']) else ""
        return f"{st_num}-{st_num2}" if st_num and st_num2 else st_num
    except:
        return ""

assessment_df["ST_NUM_COMBINED"] = assessment_df.apply(combine_street_numbers, axis=1)


In [38]:
assessment_df["ZIP_CODE_CLEAN"] = assessment_df["ZIP_CODE"].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

assessment_df["ST_NAME_CLEAN"] = assessment_df["ST_NAME"].astype(str).str.replace(r'\bAVE\.', 'AV', regex=True)

assessment_df["formatted_address"] = (
    assessment_df["ST_NUM_COMBINED"].str.strip() + " " +
    assessment_df["ST_NAME_CLEAN"].astype(str).str.strip() + ", " +
    assessment_df["CITY"].astype(str).str.strip() + ", MA " +
    assessment_df["ZIP_CODE_CLEAN"]
).str.upper()


In [39]:
assessment_df["formatted_address"].dropna().unique()[:10]

array(['104 PUTNAM ST, EAST BOSTON, MA 02128',
       '197 LEXINGTON ST, EAST BOSTON, MA 02128',
       '199 LEXINGTON ST, EAST BOSTON, MA 02128',
       '201 LEXINGTON ST, EAST BOSTON, MA 02128',
       '203 LEXINGTON ST, EAST BOSTON, MA 02128',
       '205-207 LEXINGTON ST, EAST BOSTON, MA 02128',
       '209-211 LEXINGTON ST, EAST BOSTON, MA 02128',
       '213 LEXINGTON ST, EAST BOSTON, MA 02128',
       '215 LEXINGTON ST, EAST BOSTON, MA 02128',
       '217 LEXINGTON ST, EAST BOSTON, MA 02128'], dtype=object)

In [40]:
# Show side-by-side original and formatted addresses from a sample
assessment_df[["ST_NUM", "ST_NUM2", "ST_NAME", "CITY", "ZIP_CODE", "formatted_address"]].sample(50)


,ST_NUM,ST_NUM2,ST_NAME,CITY,ZIP_CODE,formatted_address
72920,7.0,NaN,DOUGLAS ST,SOUTH BOSTON,2127.0,"7 DOUGLAS ST, SOUTH BOSTON, MA 02127"
154288,220.0,NaN,ALLANDALE ST,CHESTNUT HILL,2467.0,"220 ALLANDALE ST, CHESTNUT HILL, MA 02467"
77514,NaN,NaN,George ST,ROXBURY,2119.0,"GEORGE ST, ROXBURY, MA 02119"
176919,32.0,NaN,ARDEN ST,ALLSTON,2134.0,"32 ARDEN ST, ALLSTON, MA 02134"
674,86.0,NaN,WORDSWORTH ST,EAST BOSTON,2128.0,"86 WORDSWORTH ST, EAST BOSTON, MA 02128"
48597,145.0,NaN,Pinckney ST,BOSTON,2114.0,"145 PINCKNEY ST, BOSTON, MA 02114"
3396,104.0,NaN,Marion ST,EAST BOSTON,2128.0,"104 MARION ST, EAST BOSTON, MA 02128"
128208,137.0,NaN,SAVANNAH AV,MATTAPAN,2126.0,"137 SAVANNAH AV, MATTAPAN, MA 02126"
95247,73.0,NaN,MAYWOOD ST,ROXBURY,2119.0,"73 MAYWOOD ST, ROXBURY, MA 02119"
140859,57.0,NaN,Como RD,HYDE PARK,2136.0,"57 COMO RD, HYDE PARK, MA 02136"


# Current vs Assessor

In [41]:
# Compare formatted FY2025 addresses to registry addresses
unregistered_props = assessment_df[~assessment_df["formatted_address"].isin(registry_df["formatted_address"])]

# Count how many are unregistered
unregistered_count = unregistered_props.shape[0]

# Total properties in FY2025
total_properties = assessment_df.shape[0]

# Calculate percentage unregistered
unregistered_pct = (unregistered_count / total_properties) * 100

unregistered_count, total_properties, round(unregistered_pct, 2)

(134491, 183445, 73.31)

In [42]:
registered_properties_from_assessment = assessment_df[
    assessment_df["formatted_address"].isin(registry_df["formatted_address"])
]

# Show first 10 matching addresses
registered_properties_from_assessment["formatted_address"].sample(30)

14114             368 MAIN ST, CHARLESTOWN, MA 02129
71037      275 OLD COLONY AV, SOUTH BOSTON, MA 02127
30315               151 TREMONT ST, BOSTON, MA 02111
147117      3940 WASHINGTON ST, ROSLINDALE, MA 02131
119843          18 ROSEDALE ST, DORCHESTER, MA 02124
99812            8 PLEASANT ST, DORCHESTER, MA 02125
2817           75 WALDEMAR AV, EAST BOSTON, MA 02128
110820           7 GLENDALE ST, DORCHESTER, MA 02125
47215                 69 REVERE ST, BOSTON, MA 02114
64329                50 LIBERTY DR, BOSTON, MA 02210
49812           199 MARLBOROUGH ST, BOSTON, MA 02116
179434            361 FANEUIL ST, BRIGHTON, MA 02135
126176                21 RICH ST, MATTAPAN, MA 02126
99341           102 HANCOCK ST, DORCHESTER, MA 02125
107094             42 ESTELLA ST, MATTAPAN, MA 02126
57016            64 QUEENSBERRY ST, BOSTON, MA 02215
182102      2003 COMMONWEALTH AV, BRIGHTON, MA 02135
136433           54 PRESCOTT ST, HYDE PARK, MA 02136
170856         191 WASHINGTON ST, BRIGHTON, MA

# 311 Service Request

In [43]:
service_df = pd.read_csv("dff4d804-5031-443a-8409-8344efd0e5c8.csv", low_memory=False)

In [44]:
# Known city/neighborhood names to catch multi-word places like "SOUTH BOSTON", "JAMAICA PLAIN"
boston_neighborhoods = [
    "SOUTH BOSTON", "EAST BOSTON", "JAMAICA PLAIN", "MATTAPAN", "ROXBURY", 
    "BRIGHTON", "CHARLESTOWN", "HYDE PARK", "DORCHESTER", "WEST ROXBURY", 
    "ALLSTON", "ROSLINDALE", "BACK BAY", "FENWAY", "MISSION HILL", "NORTH END",
    "SOUTH END", "CHINATOWN"
]

def format_location(location):
    try:
        parts = location.strip().split()
        if len(parts) < 4:
            return location.upper()

        zip_code = parts[-1]
        state = parts[-2]

        # Try 2-word city names first
        possible_city = " ".join(parts[-4:-2]).upper()
        if possible_city in boston_neighborhoods:
            city = possible_city
            street = " ".join(parts[:-4])
        else:
            # Fall back to 1-word city names
            city = parts[-3].upper()
            street = " ".join(parts[:-3])

        return f"{street}, {city}, {state} {zip_code}".upper()
    except:
        return ""
service_df["formatted_address"] = service_df["location"].astype(str).apply(format_location)

# Replace AVE (or AVE.) with AV in 311 formatted addresses
service_df["formatted_address"] = service_df["formatted_address"].str.replace(
    r"\bAVE\.?\b", "AV", regex=True
)


service_df["formatted_address"].sample(30)

238381    INTERSECTION OF CLARENDON ST & STUART, ST, BOS...
242965                                                     
242076                       42 IRVING ST, BOSTON, MA 02114
264318                     320 WARREN ST, ROXBURY, MA 02119
142407    INTERSECTION OF COOLIDGE AV & CAMBRIDGE, ST, B...
91077                    200 CLARENDON ST, BOSTON, MA 02116
61185     INTERSECTION OF LONG AV & COMMONWEALTH, AV, AL...
148690                      80 F ST, SOUTH BOSTON, MA 02127
93442                    14 DITSON ST, DORCHESTER, MA 02122
118066    INTERSECTION OF W SPRINGFIELD ST & TREMONT, ST...
187537                      79 CHARLES ST, BOSTON, MA 02114
41223         681H-686 E SEVENTH ST, SOUTH BOSTON, MA 02127
157033    INTERSECTION OF CHAMPNEY PL & ANDERSON, ST, BO...
14625                     71A CHANDLER ST, BOSTON, MA 02116
71500              56 SIGOURNEY ST, JAMAICA PLAIN, MA 02130
123498              694 MASSACHUSETTS AV, ROXBURY, MA 02118
265386                39-41 BELDEN ST, D

# Compare Current vs 311 

In [45]:
unregistered_service_requests = service_df[~service_df["formatted_address"].isin(registry_df["formatted_address"])]

unregistered_service_count = unregistered_service_requests.shape[0]
total_service_requests = service_df.shape[0]
unregistered_service_pct = (unregistered_service_count / total_service_requests) * 100

unregistered_service_count, total_service_requests, round(unregistered_service_pct, 2)


(229914, 282836, 81.29)

In [46]:
registered_service_requests = service_df[service_df["formatted_address"].isin(registry_df["formatted_address"])]

registered_service_requests["formatted_address"].sample(30)

279491             58 BURRELL ST, ROXBURY, MA 02119
10806              15 ABERDEEN ST, BOSTON, MA 02215
252463     235 CHESTNUT HILL AV, BRIGHTON, MA 02135
116100      744 E FOURTH ST, SOUTH BOSTON, MA 02127
247707        32 SHEFFIELD RD, ROSLINDALE, MA 02131
1946             6 WATERLOO ST, HYDE PARK, MA 02136
148730         14 ROSSETER ST, DORCHESTER, MA 02121
47379               27 MELROSE ST, BOSTON, MA 02116
248816           19 HIAWATHA RD, MATTAPAN, MA 02126
116169         69 READVILLE ST, HYDE PARK, MA 02136
4964       161 W SEVENTH ST, SOUTH BOSTON, MA 02127
82911              119 NEWBURY ST, BOSTON, MA 02116
55592                157 SALEM ST, BOSTON, MA 02113
241192           58 SUPPLE RD, DORCHESTER, MA 02121
224990      33 BOYLSTON ST, JAMAICA PLAIN, MA 02130
67352              65 ANDERSON ST, BOSTON, MA 02114
144800            28 DARTMOUTH ST, BOSTON, MA 02116
211661        91 DRESSER ST, SOUTH BOSTON, MA 02127
257979       151 W SIXTH ST, SOUTH BOSTON, MA 02127
73532      5

# Unregistered list

In [47]:
# Extract Series of formatted addresses from each dataset
unregistered_service_addrs = service_df[~service_df["formatted_address"].isin(registry_df["formatted_address"])]["formatted_address"].dropna()

unregistered_assessor_addrs = assessment_df[~assessment_df["formatted_address"].isin(registry_df["formatted_address"])]["formatted_address"].dropna()

# Concatenate the two Series and drop duplicates
all_unregistered_addresses = pd.concat([unregistered_service_addrs, unregistered_assessor_addrs]).drop_duplicates().sort_values()

# Count
print(f"Total unique unregistered addresses: {len(all_unregistered_addresses)}")

# Preview
all_unregistered_addresses.head(10)


Total unique unregistered addresses: 110362


160                                           
65312                   A ST, BOSTON, MA 02210
57715             A ST, SOUTH BOSTON, MA 02127
175682             ABBY RD, BRIGHTON, MA 02135
181764     ACADEMY HILL RD, BRIGHTON, MA 02135
68007        ACADIA ST, SOUTH BOSTON, MA 02127
45838               ACORN ST, BOSTON, MA 02108
139861           ACTON ST, HYDE PARK, MA 02136
147890            ADA ST, ROSLINDALE, MA 02131
18398               ADAMS PL, BOSTON, MA 02114
Name: formatted_address, dtype: object